In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda"
os.makedirs(f"{PROJECT_ROOT}/data/processed", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/reports", exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Feature store exists?:", os.path.exists(f"{PROJECT_ROOT}/data/processed/trip_feature_store.parquet"))

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Porfolio/Balearia/2 Demanda
Feature store exists?: True


In [2]:
import importlib

if importlib.util.find_spec("xgboost") is None:
    !pip -q install xgboost
else:
    print("xgboost ya está instalado ✅")

xgboost ya está instalado ✅


In [3]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

df = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/trip_feature_store.parquet")
df["departure_datetime_local"] = pd.to_datetime(df["departure_datetime_local"])
df = df.sort_values("departure_datetime_local").reset_index(drop=True)

df.head()

,company,trip_id,route_id,origin_port,dest_port,departure_datetime_local,date,dep_time,weekday,month,...,pax_lag_1w,pax_lag_2w,veh_lag_1w,veh_lag_2w,pax_roll_mean_20,pax_roll_std_20,veh_roll_mean_20,veh_roll_std_20,delay_roll_mean_20,occ_pax_roll_mean_20
0,LevanteFerries,T0000075,DEN-FOR,Denia,Formentera,2024-01-07 08:00:00,2024-01-07,08:00,6,1,...,542.0,533.0,194.0,160.0,581.25,157.994295,173.80,44.684390,5.60,0.645833
1,LevanteFerries,T0000070,DEN-IBZ,Denia,Ibiza,2024-01-07 08:00:00,2024-01-07,08:00,6,1,...,759.0,548.0,235.0,170.0,664.70,148.758405,206.35,46.183017,9.35,0.738556
2,LevanteFerries,T0000069,DEN-IBZ,Denia,Ibiza,2024-01-07 12:00:00,2024-01-07,12:00,6,1,...,734.0,608.0,220.0,220.0,664.60,148.713571,205.45,45.706702,9.35,0.553833
3,LevanteFerries,T0000076,DEN-FOR,Denia,Formentera,2024-01-07 12:00:00,2024-01-07,12:00,6,1,...,631.0,528.0,214.0,162.0,587.10,156.085638,175.95,43.950301,5.60,0.652333
4,LevanteFerries,T0000077,DEN-FOR,Denia,Formentera,2024-01-07 17:00:00,2024-01-07,17:00,6,1,...,591.0,399.0,168.0,114.0,585.10,155.248494,174.35,43.221918,6.40,0.650111


In [4]:
cutoff = pd.Timestamp("2025-07-01")

train = df[df["departure_datetime_local"] < cutoff].copy()
test  = df[df["departure_datetime_local"] >= cutoff].copy()

print("Train:", train.shape, " Test:", test.shape)

Train: (6522, 36)  Test: (2346, 36)


In [5]:
feature_cols = [
    "weekday", "month",
    "capacity_pax",
    "avg_ticket_price", "price_index",
    "is_holiday_proxy", "sea_bad_proxy",
    "pax_lag_1w", "pax_lag_2w",
    "pax_roll_mean_20", "pax_roll_std_20",
    "delay_roll_mean_20",
]

X_train = train[feature_cols]
y_train = train["pax_real"]

X_test  = test[feature_cols]
y_test  = test["pax_real"]

In [7]:
model = XGBRegressor(
    n_estimators=700,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model trained ✅")

Model trained ✅


In [8]:
def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    return np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom

pred = model.predict(X_test)

# Cap por capacidad (regla de negocio: no puedes vender más pax que plazas)
pred = np.minimum(pred, test["capacity_pax"].values)
pred = np.clip(pred, 0, None)

metrics_ml = {
    "cutoff": str(cutoff.date()),
    "MAE": float(np.mean(np.abs(y_test.values - pred))),
    "WAPE": float(wape(y_test.values, pred)),
    "n_test": int(len(test))
}
metrics_ml

{'cutoff': '2025-07-01',
 'MAE': 66.50269114046348,
 'WAPE': 0.09519937944366753,
 'n_test': 2346}

In [9]:
test_out = test.copy()
test_out["pax_pred_ml"] = pred

per_route_ml = (
    test_out.groupby("route_id")
    .apply(lambda g: pd.Series({
        "MAE": np.mean(np.abs(g["pax_real"] - g["pax_pred_ml"])),
        "WAPE": wape(g["pax_real"].values, g["pax_pred_ml"].values),
        "n": len(g)
    }))
    .reset_index()
    .sort_values("WAPE")
)

display(per_route_ml)

# Guardados
test_out.to_parquet(f"{PROJECT_ROOT}/data/processed/ml_predictions.parquet", index=False)
per_route_ml.to_csv(f"{PROJECT_ROOT}/reports/ml_per_route_metrics.csv", index=False)

print("Saved ✅ ml_predictions.parquet + ml_per_route_metrics.csv")

/tmp/ipykernel_172/1157108738.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,route_id,MAE,WAPE,n
1,DEN-IBZ,65.095987,0.081817,665.0
0,DEN-FOR,66.779917,0.093419,665.0
3,VAL-IBZ,58.489920,0.105083,508.0
2,DEN-PMI,75.994012,0.109796,508.0


Saved ✅ ml_predictions.parquet + ml_per_route_metrics.csv


In [10]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
display(importances.to_frame("importance"))

importances.to_csv(f"{PROJECT_ROOT}/reports/ml_feature_importances.csv")
print("Saved ✅ reports/ml_feature_importances.csv")

,importance
pax_roll_mean_20,0.393047
weekday,0.159711
sea_bad_proxy,0.105064
is_holiday_proxy,0.074148
capacity_pax,0.071175
price_index,0.041333
pax_lag_1w,0.040061
pax_lag_2w,0.035869
pax_roll_std_20,0.022906
month,0.019665


Saved ✅ reports/ml_feature_importances.csv
